In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
import yfinance as yf

ticker = "GC=F"

gold_data = yf.download(ticker, start="2010-01-01", end=None, progress=False)

YF.download() has changed argument auto_adjust default to True


In [3]:
gold_data.reset_index(inplace=True)

In [4]:
gold_data.columns = [col[0] for col in gold_data.columns]

In [5]:
gold_data

,Date,Close,High,Low,Open,Volume
0,2010-01-04,1117.699951,1122.300049,1097.099976,1117.699951,184
1,2010-01-05,1118.099976,1126.500000,1115.000000,1118.099976,53
2,2010-01-06,1135.900024,1139.199951,1120.699951,1135.900024,363
3,2010-01-07,1133.099976,1133.099976,1129.199951,1133.099976,56
4,2010-01-08,1138.199951,1138.199951,1122.699951,1138.199951,54
...,...,...,...,...,...,...
3840,2025-04-11,3222.199951,3235.000000,3182.100098,3182.100098,862
3841,2025-04-14,3204.800049,3228.800049,3194.500000,3215.500000,263
3842,2025-04-15,3218.699951,3218.699951,3214.000000,3216.000000,390
3843,2025-04-16,3326.600098,3334.899902,3238.300049,3238.300049,1874


In [6]:
gold_data["Pct_Change"] = gold_data["Close"].pct_change()

In [7]:
# Drop missing % Change
gold_data = gold_data.dropna(subset=["Pct_Change"]).reset_index(drop=True)

# Optionally scale for better convergence (can skip this for raw % change too)
scaler = StandardScaler()
gold_data["Pct_Change"] = scaler.fit_transform(gold_data[["Pct_Change"]])

In [8]:
gold_data

,Date,Close,High,Low,Open,Volume,Pct_Change
0,2010-01-05,1118.099976,1126.500000,1115.000000,1118.099976,53,0.002446
1,2010-01-06,1135.900024,1139.199951,1120.699951,1135.900024,363,1.547452
2,2010-01-07,1133.099976,1133.099976,1129.199951,1133.099976,56,-0.277818
3,2010-01-08,1138.199951,1138.199951,1122.699951,1138.199951,54,0.413767
4,2010-01-11,1150.699951,1161.199951,1143.000000,1150.699951,177,1.057239
...,...,...,...,...,...,...,...
3839,2025-04-11,3222.199951,3235.000000,3182.100098,3182.100098,862,2.075117
3840,2025-04-14,3204.800049,3228.800049,3194.500000,3215.500000,263,-0.569202
3841,2025-04-15,3218.699951,3218.699951,3214.000000,3216.000000,390,0.397515
3842,2025-04-16,3326.600098,3334.899902,3238.300049,3238.300049,1874,3.295088


In [9]:
SEQ_LEN = 10  # use past 10 days to predict next day


class LaggedPctChangeDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i : i + seq_len])  # t-10 to t-1
            self.y.append(series[i + seq_len])  # t
        self.X = torch.tensor(self.X, dtype=torch.float32).unsqueeze(-1)  # (N, 10, 1)
        self.y = torch.tensor(self.y, dtype=torch.float32).unsqueeze(1)  # (N, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
# Get the % Change series
series = gold_data["Pct_Change"].values

# Split preserving order
split_idx = int(len(series) * 0.8)
train_series = series[:split_idx]
test_series = series[split_idx - SEQ_LEN :]  # keep overlap

# Datasets and loaders
train_dataset = LaggedPctChangeDataset(train_series, SEQ_LEN)
test_dataset = LaggedPctChangeDataset(test_series, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

/tmp/ipykernel_15826/1304005806.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.X = torch.tensor(self.X, dtype=torch.float32).unsqueeze(-1)  # (N, 10, 1)


In [11]:
class PctChangeLSTM(nn.Module):
    def __init__(self, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # take last time step's output

In [12]:
model = PctChangeLSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [13]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Train Loss: {total_loss / len(train_loader):.4f}")

Epoch 1 | Train Loss: 1.0402
Epoch 2 | Train Loss: 1.0392
Epoch 3 | Train Loss: 1.0391
Epoch 4 | Train Loss: 1.0390
Epoch 5 | Train Loss: 1.0389
Epoch 6 | Train Loss: 1.0388
Epoch 7 | Train Loss: 1.0386
Epoch 8 | Train Loss: 1.0385
Epoch 9 | Train Loss: 1.0381
Epoch 10 | Train Loss: 1.0371
Epoch 11 | Train Loss: 1.0378
Epoch 12 | Train Loss: 1.0357
Epoch 13 | Train Loss: 1.0340
Epoch 14 | Train Loss: 1.0365
Epoch 15 | Train Loss: 1.0378
Epoch 16 | Train Loss: 1.0359
Epoch 17 | Train Loss: 1.0330
Epoch 18 | Train Loss: 1.0310
Epoch 19 | Train Loss: 1.0281
Epoch 20 | Train Loss: 1.0288
